## Bronx condos (rolling sales)

This notebook mirrors the base condo analysis notebooks, but focuses on **Bronx** condo sales (rows where `BUILDING CLASS CATEGORY` contains `CONDO` and does **not** contain `COOP`).


In [1]:
# install the dependencies (pandas, numpy, openpyxl for .xlsx, ...)
%pip install pandas numpy openpyxl matplotlib seaborn scikit-learn -q

# import the dependencies
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Rolling sales by borough: one Excel file per NYC borough under ./data
DATA_DIR = Path("data")

# (display name, filename)
BOROUGH_FILES = [
    ("Bronx", "rollingsales_bronx.xlsx"),
    ("Brooklyn", "rollingsales_brooklyn.xlsx"),
    ("Manhattan", "rollingsales_manhattan.xlsx"),
    ("Queens", "rollingsales_queens.xlsx"),
    ("Staten Island", "rollingsales_statenisland.xlsx"),
]

borough_frames: dict[str, pd.DataFrame] = {}
for borough, fname in BOROUGH_FILES:
    path = DATA_DIR / fname
    # First 4 rows are title/description; real table header starts on row 5
    df = pd.read_excel(path, skiprows=4)
    borough_frames[borough] = df

# Remove any rows where the sale price is 0 or negative or missing
for borough in borough_frames:
    df = borough_frames[borough]
    df = df[df["SALE PRICE"].notna()]
    df = df[df["SALE PRICE"] > 0]
    borough_frames[borough] = df

# Dropping irrelevant columns
columns_to_drop = ["BOROUGH", "APARTMENT NUMBER", "EASEMENT"]
for borough in borough_frames:
    try:
        borough_frames[borough] = borough_frames[borough].drop(columns=columns_to_drop)
    except KeyError:
        pass

sales = pd.concat(borough_frames.values(), ignore_index=True)
print("Combined sales shape:", sales.shape)


In [ ]:
# How empty is each column in each borough? + simple unit imputation

def is_nullish(s: pd.Series) -> pd.Series:
    null = s.isna()
    if s.dtype != object and not pd.api.types.is_string_dtype(s):
        return null
    empty_str = s.map(lambda x: isinstance(x, str) and x.strip() == "")
    return null | empty_str

# Fill in missing values for residential, commercial, and total units
for borough in borough_frames:
    df = borough_frames[borough]
    for idx, row in df.iterrows():
        residential_units = row["RESIDENTIAL UNITS"]
        commercial_units = row["COMMERCIAL UNITS"]
        total_units = row["TOTAL UNITS"]

        r_ok = pd.notna(residential_units) and not (
            isinstance(residential_units, str) and residential_units.strip() == ""
        )
        c_ok = pd.notna(commercial_units) and not (
            isinstance(commercial_units, str) and commercial_units.strip() == ""
        )
        t_ok = pd.notna(total_units) and not (
            isinstance(total_units, str) and total_units.strip() == ""
        )

        if not r_ok and t_ok and c_ok:
            residential_units = total_units - commercial_units
        elif not t_ok and r_ok and c_ok:
            total_units = residential_units + commercial_units
        elif not c_ok and t_ok and r_ok:
            commercial_units = total_units - residential_units

        df.loc[idx, "RESIDENTIAL UNITS"] = residential_units
        df.loc[idx, "COMMERCIAL UNITS"] = commercial_units
        df.loc[idx, "TOTAL UNITS"] = total_units

# Rebuild combined frame so later analysis sees imputed unit columns
sales = pd.concat(borough_frames.values(), ignore_index=True)

# Quick missingness report for Bronx only
bx = borough_frames["Bronx"]
print("Bronx column empty fraction:")
for col in bx.columns:
    print(f"{col}: {is_nullish(bx[col]).mean():.4f}")


In [ ]:
# Bronx condos (exclude categories that also contain COOP, e.g. CONDO COOPS)
_bcc = borough_frames["Bronx"]["BUILDING CLASS CATEGORY"].astype(str).str.upper()
bronx_condos = borough_frames["Bronx"][_bcc.str.contains("CONDO") & ~_bcc.str.contains("COOP")].copy()

print("Total number of condo sales (Bronx, CONDO & ~COOP):")
print(bronx_condos.shape[0])

print("Building class categories:")
print(pd.Series(bronx_condos["BUILDING CLASS CATEGORY"].unique()).sort_values().to_list())

print("Columns:")
print(bronx_condos.columns)


In [ ]:
# Drop all-null columns in this slice, then drop LAND/GROSS SF (matches other base condo notebooks)
print("All-null columns in Bronx condo slice:")
print(bronx_condos.columns[bronx_condos.isna().all()])

bronx_condos = bronx_condos.dropna(axis=1, how="all")

print("Null fraction in Bronx condo slice:")
for col in bronx_condos.columns:
    print(f"{col}: {bronx_condos[col].isna().mean():.4f}")

for c in ("LAND SQUARE FEET", "GROSS SQUARE FEET"):
    if c in bronx_condos.columns:
        bronx_condos = bronx_condos.drop(columns=[c])

bronx_condos.head()


In [ ]:
# Time & place: walkup vs elevator, neighborhood, vintage (condo slice)
REF_YEAR = 2026

b = borough_frames["Bronx"].copy()
_bcc = b["BUILDING CLASS CATEGORY"].astype(str).str.upper()
condo_tp = b[_bcc.str.contains("CONDO") & ~_bcc.str.contains("COOP")].copy()

condo_tp["SALE DATE"] = pd.to_datetime(condo_tp["SALE DATE"])

bcc_upper = condo_tp["BUILDING CLASS CATEGORY"].astype(str).str.upper()
condo_tp["BUILDING_STYLE"] = np.where(
    bcc_upper.str.contains("WALKUP"),
    "Walkup",
    np.where(bcc_upper.str.contains("ELEVATOR"), "Elevator", "Other"),
)

condo_tp["YEAR BUILT"] = pd.to_numeric(condo_tp["YEAR BUILT"], errors="coerce")
condo_tp["BUILDING_AGE"] = REF_YEAR - condo_tp["YEAR BUILT"]

# Upper-tail IQR trim on sale price (keeps bulk of distribution, drops extreme outliers)
q1 = condo_tp["SALE PRICE"].quantile(0.25)
q3 = condo_tp["SALE PRICE"].quantile(0.75)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
condo_tp = condo_tp[condo_tp["SALE PRICE"] < upper_fence].copy()

condo_tp["YEAR"] = condo_tp["SALE DATE"].dt.year
condo_tp["MONTH"] = condo_tp["SALE DATE"].dt.month
condo_tp["YEAR_MONTH"] = condo_tp["SALE DATE"].dt.to_period("M")
condo_tp["QUARTER"] = condo_tp["SALE DATE"].dt.to_period("Q")

print("BUILDING_STYLE counts:\n", condo_tp["BUILDING_STYLE"].value_counts(), sep="")
print(f"\nRows after upper IQR trim: {len(condo_tp):,}")
print(f"Sale date range: {condo_tp['SALE DATE'].min().date()} – {condo_tp['SALE DATE'].max().date()}")
print(f"Neighborhoods: {condo_tp['NEIGHBORHOOD'].nunique()}")


In [ ]:
# Comparable outcomes: total sale price vs price per residential unit; deal scope; BCC segment
ru = pd.to_numeric(condo_tp["RESIDENTIAL UNITS"], errors="coerce")
sp = condo_tp["SALE PRICE"].astype(float)
condo_tp["PRICE_PER_RES_UNIT"] = np.where(ru > 0, sp / ru, np.nan)
condo_tp["DEAL_SCOPE"] = np.where(
    ru.isna() | (ru <= 0),
    "Unknown/missing units",
    np.where(ru == 1, "Single unit (likely apartment sale)", "Multi-unit / bulk"),
)

bcc = condo_tp["BUILDING CLASS CATEGORY"].astype(str).str.upper()
cond_special = bcc.str.contains("SPECIAL CONDO BILLING")
cond_comm = bcc.str.contains(
    "CONDO STORE|CONDO OFFICE|CONDO HOTEL|COMMERCIAL CONDO|CONDO PARKING|WAREHOUSE|FACTORY|"
    "NON-BUSINESS STORAGE|TERRACES|CULTURAL|MEDICAL|EDUCATIONAL"
)
cond_res = bcc.str.contains("WALKUP|ELEVATOR|2-10 UNIT|TAX CLASS 1 CONDOS")
condo_tp["BCC_SEGMENT"] = np.select(
    [cond_special, cond_comm, cond_res],
    ["Special billing lots", "Commercial / non-housing", "Residential (walkup/elevator/small)"],
    default="Other / unclassified",
)

print("DEAL_SCOPE:\n", condo_tp["DEAL_SCOPE"].value_counts(), sep="")
print("\nBCC_SEGMENT:\n", condo_tp["BCC_SEGMENT"].value_counts(), sep="")
